# Trial Inspector

Set one `generation_run_id` + `trial_id` below and re-run every cell (Run All) to see
everything about that trial in one place: the canonical scene/character data, the U/S/P
dialogue transcripts, the H1 judge prompts (with a check for whether they're really
condition-blind), the full H1 metric breakdown per condition, and the H2 realism
judge's critique, scores, and score explanation per condition.

This is a drill-down tool for one trial at a time, separate from `results_dashboard.ipynb`
(which aggregates across trials/conditions).

In [172]:
import glob
import json
from pathlib import Path

REPO_ROOT = Path.cwd().parent  # this notebook lives in notebooks/, one level under the repo root
H1_PATH = REPO_ROOT / "output" / "evaluation" / "h1_consistency.jsonl"
H2_PATH = REPO_ROOT / "output" / "evaluation" / "h2_realism.jsonl"
TRAJECTORY_GLOB = str(REPO_ROOT / "output" / "record" / "*" / "trajectory" / "*.jsonl")
ALL_ACTIONS_GLOB = str(REPO_ROOT / "output" / "record" / "*" / "all" / "*.jsonl")

CONDITION_ORDER = ["U", "S", "P"]


def load_jsonl(path):
    with open(path, encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]

In [173]:
def find_trajectory_records(generation_run_id, trial_id):
    """Search every trajectory file under output/record/*/trajectory/, not one hardcoded
    path, so this keeps working if more scenes/titles/models get added later."""
    matches = {}
    for path in sorted(glob.glob(TRAJECTORY_GLOB)):
        for record in load_jsonl(path):
            if record.get("generation_run_id") == generation_run_id and record.get("trial_id") == trial_id:
                matches[record["context_condition"]] = record
    return matches


def find_generation_calls(generation_run_id, trial_id, condition):
    """One line in the `all` record file per (trial, condition), saved once the whole
    trial finishes; its `record` list holds one {prompt, response, duration} dict per
    round, in round order (record[0] = round 1). Search every `all` file, not one
    hardcoded path, for the same reason as find_trajectory_records()."""
    for path in sorted(glob.glob(ALL_ACTIONS_GLOB)):
        for line in load_jsonl(path):
            if (
                line.get("generation_run_id") == generation_run_id
                and line.get("trial_id") == trial_id
                and line.get("context_condition") == condition
            ):
                return line["record"]
    return None


def find_h1_rows(generation_run_id, trial_id):
    by_condition = {condition: [] for condition in CONDITION_ORDER}
    for row in load_jsonl(H1_PATH):
        if row.get("generation_run_id") == generation_run_id and row.get("trial_id") == trial_id:
            by_condition[row["context_condition"]].append(row)
    for condition in by_condition:
        by_condition[condition].sort(key=lambda r: r["round"])
    return by_condition


def find_h2_rows(generation_run_id, trial_id):
    """One H2 row per (trial, condition): H2 scores the whole trajectory at once, not
    per round. Returns None for a condition with no H2 row yet."""
    by_condition = {condition: None for condition in CONDITION_ORDER}
    if not H2_PATH.exists():
        return by_condition
    for row in load_jsonl(H2_PATH):
        if row.get("generation_run_id") == generation_run_id and row.get("trial_id") == trial_id:
            by_condition[row["context_condition"]] = row
    return by_condition


## Set the trial to inspect

In [ ]:
GENERATION_RUN_ID = "cb_hp_019_luna_neville-1790690383028011000"
TRIAL_ID = "cb_hp_019_luna_neville_t11"

trajectory_records = find_trajectory_records(GENERATION_RUN_ID, TRIAL_ID)
h1_rows = find_h1_rows(GENERATION_RUN_ID, TRIAL_ID)
generation_calls = {c: find_generation_calls(GENERATION_RUN_ID, TRIAL_ID, c) for c in CONDITION_ORDER}
h2_rows = find_h2_rows(GENERATION_RUN_ID, TRIAL_ID)

missing_trajectory = [c for c in CONDITION_ORDER if c not in trajectory_records]
missing_h1 = [c for c in CONDITION_ORDER if not h1_rows[c]]
missing_generation = [c for c in CONDITION_ORDER if not generation_calls[c]]
if missing_trajectory:
    raise ValueError(f"No trajectory record found for conditions: {missing_trajectory}")
if missing_h1:
    raise ValueError(f"No H1 rows found for conditions: {missing_h1}")
if missing_generation:
    raise ValueError(f"No generation-call trace ('all' record) found for conditions: {missing_generation}")

print(f"Found trial {TRIAL_ID!r} in run {GENERATION_RUN_ID!r}: all of U/S/P present.")
print({c: len(h1_rows[c]) for c in CONDITION_ORDER}, "rounds scored per condition")
missing_h2 = [c for c in CONDITION_ORDER if h2_rows[c] is None]
if missing_h2:
    # Not an error: H2 sections 9-11 just report the gap, the H1 sections still work.
    print(f"No H2 row yet for conditions: {missing_h2}")


Found trial 'cb_hp_012_newt_mcgonagall_t11' in run 'cb_hp_012_newt_mcgonagall-1790692368262895000': all of U/S/P present.
{'U': 10, 'S': 10, 'P': 10} rounds scored per condition


### 1. Scene information

In [175]:
reference = trajectory_records["U"]
scene = reference["scene"]
player = reference["player"]
target = reference["target"]

print("[Scene]")
for key in ("event", "time", "location", "description"):
    print(f"{key.capitalize()}: {scene.get(key)}")

print()
print(f"[Player — {player['name']}]")
for key in ("position", "states", "description"):
    print(f"{key.capitalize()}: {player.get(key)}")

print()
print(f"[Target — {target['name']}]")
for key in ("position", "states", "description", "self_belief", "environment_belief"):
    print(f"{key.capitalize()}: {target.get(key)}")

print()
print("[Action Reference]")
print(reference["action_reference"])

# docs/research-contract.md §5: fields-and-values(U) = fields-and-values(S) = fields-and-values(P).
# Cheap, concrete check that this trial actually respects it, not just an assumption.
equivalent = all(
    (trajectory_records[c]["scene"], trajectory_records[c]["target"], trajectory_records[c]["action_reference"])
    == (scene, target, reference["action_reference"])
    for c in CONDITION_ORDER
)
print()
print(f"Scene/character/action_reference identical across U/S/P: {equivalent}")

[Scene]
Event: The Escape from Punishment
Time: Late Morning
Location: The Great Hall, Hogwarts
Description: A scene unfolds in the Great Hall where a student is caught in a compromising situation. The atmosphere is tense, with onlookers and the weather adding to the drama.

[Player — Newt Scamander]
Position: Standing near the Slytherin table
States: Nervous
Description: A Magizoologist and former Hogwarts student, known for his expertise with magical creatures.

[Target — Professor McGonagall]
Position: At the front of the Great Hall, ready to discipline the student
States: Annoyed
Description: The Transfiguration professor and Head of Gryffindor House, known for her strict but fair nature.
Self_belief: 
Environment_belief: 

[Action Reference]
Begins to scold Newt for the disruption.

Scene/character/action_reference identical across U/S/P: True


In [176]:
def format_interaction(record):
    player_name = record["player"]["name"]
    target_name = record["target"]["name"]
    lines = []
    for turn in record["turns"]:
        lines.append(f"Round {turn['round']}")
        lines.append(f"{player_name}: {turn['player_dialogue']}")
        lines.append(f"{target_name}: {turn['npc_dialogue']}")
    return "\n".join(lines)


def print_generation_prompt_structure(condition):
    """Round 1's actual filled generation prompt for this condition -- this is where
    U/S/P genuinely differ in structure (unlike the H1 judge prompt, which is
    deliberately condition-blind). Later rounds follow the same structure with the
    rolling Observation/memory content updated each round; not reprinted per round
    here since the structure itself doesn't change, only that one field's content."""
    first_call = generation_calls[condition][0]
    print(first_call["prompt"])

### 2. U — interaction

**Prompt structure used to generate U's responses** (round 1, representative):

In [177]:
print_generation_prompt_structure("U")

Please act as Professor McGonagall and respond using the tone, manner and vocabulary Professor McGonagall would use. Use the following context to produce one appropriate line of dialogue.
The Escape from Punishment
Late Morning
The Great Hall, Hogwarts
A scene unfolds in the Great Hall where a student is caught in a compromising situation. The atmosphere is tense, with onlookers and the weather adding to the drama.
Professor McGonagall
The Transfiguration professor and Head of Gryffindor House, known for her strict but fair nature.
At the front of the Great Hall, ready to discipline the student
Annoyed



Previous interaction:
Newt Scamander:Tries to explain the situation involving a magical creature that escaped from his care.;Newt Scamander:It was just an accident with a Niffler!
Professor McGonagall:Begins to scold Newt for the disruption.;Professor McGonagall:Scamander, this is the last straw!
Current input:
Newt Scamander: "Professor, a Ministry official, Madam Oakhollow, is comin

In [178]:
print(format_interaction(trajectory_records["U"]))

Round 1
Newt Scamander: Professor, a Ministry official, Madam Oakhollow, is coming to collect the Niffler at four o'clock.
Professor McGonagall: Mr. Scamander, while I appreciate your attempt to rectify this situation, I must remind you that accidents involving magical creatures do not absolve you from the consequences of your recklessness. You will attend to this Niffler immediately, and then we shall discuss your rather alarming tendency to invite chaos into my Great Hall.
Round 2
Newt Scamander: Yes, Professor. You should also know it took six gold coins from the Ravenclaw table.
Professor McGonagall: Mr. Scamander, I trust you understand that the theft of gold from the Ravenclaw table is hardly an acceptable form of creature management. I suggest you retrieve those coins posthaste, lest you find yourself in need of a rather stern lesson in responsibility.
Round 3
Newt Scamander: What did you see when you first came into the hall this morning, Professor?
Professor McGonagall: Mr. Sc

### 3. S — interaction

**Prompt structure used to generate S's responses** (round 1, representative):

In [179]:
print_generation_prompt_structure("S")

Please act as Professor McGonagall  and respond using the tone, manner and vocabulary Professor McGonagall would use.
 The context and Professor McGonagall's information is as follows:
Event: The Escape from Punishment
Time: Late Morning
Location: The Great Hall, Hogwarts
Description: A scene unfolds in the Great Hall where a student is caught in a compromising situation. The atmosphere is tense, with onlookers and the weather adding to the drama.
Character Profile:
Name: Professor McGonagall
Character Description: The Transfiguration professor and Head of Gryffindor House, known for her strict but fair nature.
Position: At the front of the Great Hall, ready to discipline the student
Current State: Annoyed
Self Belief: 
Environment Belief: 
Recent Memory(Ordered in reverse chronological order): 
Observation: Previous interaction:
Newt Scamander:Tries to explain the situation involving a magical creature that escaped from his care.;Newt Scamander:It was just an accident with a Niffler!


In [180]:
print(format_interaction(trajectory_records["S"]))

Round 1
Newt Scamander: Professor, a Ministry official, Madam Oakhollow, is coming to collect the Niffler at four o'clock.
Professor McGonagall: Mr. Scamander, while I appreciate your attempts to rectify this situation, I must remind you that your penchant for chaos does not excuse the disruption caused within these hallowed halls. You are to report to my office immediately after lunch, and I expect a full account of how this Niffler came to be roaming freely in the Great Hall. I daresay Madam Oakhollow will not be pleased, and I would prefer to avoid further complications.
Round 2
Newt Scamander: Yes, Professor. You should also know it took six gold coins from the Ravenclaw table.
Professor McGonagall: Mr. Scamander, I must insist that your penchant for mischief does not extend to pilfering from your fellow students. Six gold coins, you say? I shall expect you to return them posthaste, along with a proper apology to the Ravenclaw table. And let us not forget that a Niffler in the Grea

### 4. P — interaction

**Prompt structure used to generate P's responses** (round 1, representative):

In [181]:
print_generation_prompt_structure("P")

Please act as Professor McGonagall and respond using the tone, manner and vocabulary Professor McGonagall would use.
The context below is ordered from most important (Priority 1) to least important. Use all relevant context when it is compatible. If two pieces of context conflict, follow the one with the smaller priority number.
Priority 1 — current_world_constraints
Instruction: Must not be contradicted, and overrides all other priorities.
Time: Late Morning
Location: The Great Hall, Hogwarts
Position: At the front of the Great Hall, ready to discipline the student
Priority 2 — world_background
Instruction: Keep the response consistent with the fictional world.
Event: The Escape from Punishment
Description: A scene unfolds in the Great Hall where a student is caught in a compromising situation. The atmosphere is tense, with onlookers and the weather adding to the drama.
Priority 3 — character_guidance
Instruction: Use these fields to preserve personality, tone, role, and beliefs.
Name

In [182]:
print(format_interaction(trajectory_records["P"]))

Round 1
Newt Scamander: Professor, a Ministry official, Madam Oakhollow, is coming to collect the Niffler at four o'clock.
Professor McGonagall: Mr. Scamander, while I appreciate your attempts to explain, I must remind you that the chaos you have caused in the Great Hall is entirely unacceptable. A Niffler, of all creatures, should not be causing such a ruckus during breakfast! You will need to take full responsibility for this situation, and I expect you to ensure that Madam Oakhollow finds everything in order by four o'clock. Now, if you will excuse me, I have a school to maintain.
Round 2
Newt Scamander: Yes, Professor. You should also know it took six gold coins from the Ravenclaw table.
Professor McGonagall: Mr. Scamander, I daresay that your Niffler's penchant for shiny objects is rather less charming when it results in the pilfering of fellow students' possessions. I trust you will rectify this situation posthaste and return the gold to its rightful owners before Madam Oakhollow

### 5. Judge prompt (H1)

One prompt per round per condition. `[Scene]`/`[Character]`/`[Action Reference]` should be
identical across U/S/P (same values, per the equivalence check above); `[Prior turns]`/
`[Current turn]` are expected to differ, since that's the actual generated dialogue, which
does vary by condition. The check below only compares the part *before* `[Prior turns]`.

In [183]:
def prompt_prefix(prompt_text):
    marker = "[Prior turns]"
    index = prompt_text.find(marker)
    return prompt_text[:index] if index != -1 else prompt_text


rounds = [row["round"] for row in h1_rows["U"]]
for round_number in rounds:
    prompts = {
        condition: next(row["prompt"] for row in h1_rows[condition] if row["round"] == round_number)
        for condition in CONDITION_ORDER
    }
    prefixes = {condition: prompt_prefix(text) for condition, text in prompts.items()}
    identical = len(set(prefixes.values())) == 1

    print(f"===== Round {round_number} — scene/character/rubric identical across U/S/P: {identical} =====")
    for condition in CONDITION_ORDER:
        print(f"--- {condition} ---")
        print(prompts[condition])
        print()

===== Round 1 — scene/character/rubric identical across U/S/P: True =====
--- U ---
Please evaluate one turn of Professor McGonagall's dialogue against the supplied scene and character information, and against the character's own earlier turns in this conversation.

[Scene]
Event: The Escape from Punishment
Time: Late Morning
Location: The Great Hall, Hogwarts
Description: A scene unfolds in the Great Hall where a student is caught in a compromising situation. The atmosphere is tense, with onlookers and the weather adding to the drama.

[Character — Professor McGonagall]
Position: At the front of the Great Hall, ready to discipline the student
Current State: Annoyed
Description: The Transfiguration professor and Head of Gryffindor House, known for her strict but fair nature.

[Action Reference]
Begins to scold Newt for the disruption.

[Prior turns]
(none — this is the first turn)

[Current turn — evaluate this response only]
Round 1 — Player: "Professor, a Ministry official, Madam Oak

### 6. U — H1 metrics

In [184]:
def format_world_consistency(entry):
    """world_consistency has no top-level evidence field -- its decision is derived
    (in evaluate.py) from a claims list, so show that list rather than a single quote.
    "checked_against" is required on every claim now (shown as "conflicts with" for an
    inconsistent claim, "checked against" for a consistent one, including the literal
    "No supplied fact addresses this claim." placeholder when nothing applies). See the
    2026-09-16 provenance entries (claim-list redesign, checked_against rename, then
    required-on-every-claim).

    Each claim also carries "quote" (the verbatim response fragment it's drawn from)
    and "quote_verified" (whether that fragment actually appears in the NPC's real
    dialogue) -- a False here means the judge's "claim" may be misrepresenting what was
    actually said, worth checking by hand. See the 2026-09-23 provenance entry."""
    lines = [f"World Consistency: {entry['decision']}"]
    if not entry["claims"]:
        lines.append("  (no checkable claims)")
    for claim in entry["claims"]:
        verified_flag = "" if claim.get("quote_verified", True) else "  [UNVERIFIED QUOTE]"
        if claim["consistent"]:
            lines.append(f"  [consistent] {claim['claim']}{verified_flag}")
            lines.append(f"    checked against: {claim['checked_against']}")
        else:
            lines.append(f"  [CONTRADICTS] {claim['claim']}{verified_flag}")
            lines.append(f"    conflicts with: {claim['checked_against']}")
        lines.append(f"    quote: {claim.get('quote', '(none)')!r}")
    return "\n".join(lines)


def format_h1_metrics(rows):
    blocks = []
    for row in rows:
        decisions = row["decisions"]
        context_use = decisions["context_use"]
        no_contradiction = decisions["no_contradiction"]
        lines = [
            f"Round {row['round']}",
            f"Context Use: {context_use['decision']}",
            f"{context_use['evidence'] or '(none — N/A)'}",
            format_world_consistency(decisions["world_consistency"]),
            f"No Contradiction: {no_contradiction['decision']}",
            f"Contradicted round: {no_contradiction['contradicted_round'] if no_contradiction['contradicted_round'] is not None else '(none)'}",
            f"{no_contradiction['evidence'] or '(none — N/A)'}",
        ]
        blocks.append("\n".join(lines))
    return "\n\n".join(blocks)


print(format_h1_metrics(h1_rows["U"]))

Round 1
Context Use: PASS
Elicited fact: Action Reference = Begins to scold Newt for the disruption. Response: 'I must remind you that accidents involving magical creatures do not absolve you from the consequences of your recklessness.'
World Consistency: PASS
  [consistent] Professor McGonagall appreciates Newt's attempt to rectify the situation
    checked against: No supplied fact addresses this claim.
    quote: 'Mr. Scamander, while I appreciate your attempt to rectify this situation'
  [consistent] accidents involving magical creatures do not absolve Newt from the consequences of his recklessness
    checked against: No supplied fact addresses this claim.
    quote: 'accidents involving magical creatures do not absolve you from the consequences of your recklessness'
  [consistent] Newt will attend to the Niffler immediately
    checked against: No supplied fact addresses this claim.
    quote: 'You will attend to this Niffler immediately'
  [consistent] Newt has an alarming tende

### 7. S — H1 metrics

In [185]:
print(format_h1_metrics(h1_rows["S"]))

Round 1
Context Use: PASS
Elicited fact: A Ministry official, Madam Oakhollow, is coming to collect the Niffler at four o'clock. Response: 'I daresay Madam Oakhollow will not be pleased, and I would prefer to avoid further complications.'
World Consistency: PASS
  [consistent] Professor McGonagall appreciates Newt's attempts to rectify the situation
    checked against: Character description: known for her strict but fair nature.
    quote: 'Mr. Scamander, while I appreciate your attempts to rectify this situation'
  [consistent] Newt has a penchant for chaos and it does not excuse the disruption
    checked against: No supplied fact addresses this claim.
    quote: 'your penchant for chaos does not excuse the disruption caused within these hallowed halls'
  [consistent] Newt is to report to Professor McGonagall's office immediately after lunch
    checked against: Action Reference: Begins to scold Newt for the disruption.
    quote: 'You are to report to my office immediately after lu

### 8. P — H1 metrics

In [186]:
print(format_h1_metrics(h1_rows["P"]))

Round 1
Context Use: PASS
Elicited fact: Action Reference = Begins to scold Newt for the disruption. Response: 'Mr. Scamander, while I appreciate your attempts to explain, I must remind you that the chaos you have caused in the Great Hall is entirely unacceptable.'
World Consistency: PASS
  [consistent] Mr. Scamander caused chaos in the Great Hall.
    checked against: Action Reference: Begins to scold Newt for the disruption.
    quote: 'Mr. Scamander, while I appreciate your attempts to explain, I must remind you that the chaos you have caused in the Great Hall is entirely unacceptable.'
  [consistent] A Niffler caused a ruckus during breakfast.
    checked against: No supplied fact addresses this claim.
    quote: 'A Niffler, of all creatures, should not be causing such a ruckus during breakfast!'
  [consistent] Mr. Scamander needs to ensure that Madam Oakhollow finds everything in order by four o'clock.
    checked against: No supplied fact addresses this claim.
    quote: "You wil

### 9. U — H2 realism

H2 uses CharacterBox's two-call judge, so the judge's reasoning is visible in two places:

1. **Critique** — written first, in a separate call that never gives scores. This is the
   closest thing to "what the LLM thinks before scoring": it lists issues it found.
2. **Scores + explanation** — a second call receives the critique and returns the seven
   1–5 scores. Its explanation comes *after* the numbers in the same response, so it is a
   justification of scores already chosen, not reasoning that produced them.

When a score looks wrong, check the critique first: a mistaken critique (e.g. an invented
"fact" about the character) carries straight into the scores.

In [187]:
H2_METRICS = [
    "Knowledge Accuracy",
    "Emotional Expression",
    "Personality Traits",
    "Behavioral Accuracy",
    "Immersion",
    "Adaptability",
    "Behavioral Coherence",
]


def split_score_response(judge_response):
    """The score call answers "Metric: N" lines first, then (optionally) explains them.
    Everything after the last metric line is the explanation."""
    lines = judge_response.splitlines()
    last_score_line = -1
    for index, line in enumerate(lines):
        if any(line.strip().strip("*").startswith(f"{metric}:") for metric in H2_METRICS):
            last_score_line = index
    explanation = "\n".join(lines[last_score_line + 1:]).strip()
    return explanation or "(the judge gave scores only, no explanation)"


def format_h2(row):
    if row is None:
        return "(no H2 row for this condition yet — run evaluate.py on this generation run)"
    lines = [f"Judge: {row.get('judge_model')}  seed: {row.get('judge_seed')}"]
    lines.append(f"Rounds evaluated: {row.get('rounds_evaluated')}")
    lines.append("")
    lines.append("[Scores]")
    for metric in H2_METRICS:
        lines.append(f"  {metric:<22} {row['scores'].get(metric)}")
    lines.append(f"  {'Average':<22} {row['average']:.2f}")
    lines.append("")
    lines.append("[Critique — written before scoring, separate call]")
    lines.append(row["critique"].strip())
    lines.append("")
    lines.append("[Score explanation — written after the scores, same call]")
    lines.append(split_score_response(row["judge_response"]))
    return "\n".join(lines)


print(format_h2(h2_rows["U"]))

Judge: gpt-4o  seed: 1
Rounds evaluated: 10

[Scores]
  Knowledge Accuracy     1
  Emotional Expression   2
  Personality Traits     2
  Behavioral Accuracy    2
  Immersion              2
  Adaptability           1
  Behavioral Coherence   2
  Average                1.71

[Critique — written before scoring, separate call]
The role-play scenario presents several issues based on the evaluation criteria. Firstly, there is a factual inaccuracy regarding Newt Scamander's presence at Hogwarts, as he is not a student during the timeline of the Harry Potter series. Character consistency is compromised as Professor McGonagall's responses become repetitive and lack the depth expected from her character, who is known for her strict yet fair demeanor. The dialogue exhibits content redundancy, with Newt repeatedly asking about Madam Oakhollow's arrival, which detracts from engagement and realism. Emotional expression is somewhat lacking, as McGonagall's responses do not fully convey the urgency or

### 10. S — H2 realism

In [188]:
print(format_h2(h2_rows["S"]))

Judge: gpt-4o  seed: 1
Rounds evaluated: 10

[Scores]
  Knowledge Accuracy     1
  Emotional Expression   3
  Personality Traits     3
  Behavioral Accuracy    2
  Immersion              2
  Adaptability           2
  Behavioral Coherence   2
  Average                2.14

[Critique — written before scoring, separate call]
The role-play scenario presents several issues based on the evaluation criteria. Firstly, there is a factual inaccuracy regarding Newt Scamander's presence at Hogwarts, as he was not a student during the time Professor McGonagall taught there. Character consistency is compromised as Professor McGonagall's responses become repetitive and lack the depth expected from her character, who is known for her strict yet fair demeanor. Logical coherence is disrupted by Newt's repeated questions about the Niffler's collection, which seem to ignore previous answers and do not advance the narrative. Content redundancy is evident in the repetitive dialogue about Madam Oakhollow's 

### 11. P — H2 realism

Followed by a side-by-side score table for all three conditions.

In [189]:
print(format_h2(h2_rows["P"]))

print()
print("[U/S/P score comparison]")
print(f"  {'Metric':<22} " + "  ".join(f"{c:>4}" for c in CONDITION_ORDER))
for metric in H2_METRICS + ["Average"]:
    values = []
    for condition in CONDITION_ORDER:
        row = h2_rows[condition]
        if row is None:
            values.append("   -")
        elif metric == "Average":
            values.append(f"{row['average']:4.2f}")
        else:
            values.append(f"{row['scores'].get(metric):>4}")
    print(f"  {metric:<22} " + "  ".join(values))

Judge: gpt-4o  seed: 1
Rounds evaluated: 10

[Scores]
  Knowledge Accuracy     1
  Emotional Expression   2
  Personality Traits     2
  Behavioral Accuracy    2
  Immersion              2
  Adaptability           1
  Behavioral Coherence   2
  Average                1.71

[Critique — written before scoring, separate call]
The role-play scenario presents several issues based on the evaluation criteria. Firstly, there is a factual inaccuracy regarding Newt Scamander's presence at Hogwarts, as he is not a student during the timeline of the Harry Potter series. Character consistency is compromised as Professor McGonagall's responses become repetitive and lack the depth expected from her character, who is known for her sharp wit and authority. Logical coherence is disrupted by Newt's repeated questions about the same information, which seems out of character for someone knowledgeable about magical creatures. Content redundancy is evident in the repetitive dialogue about Madam Oakhollow's a